<a href="https://colab.research.google.com/github/AR-Ashik-9997/Phitron-practice-problem/blob/main/RNN_Implimentations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import torch
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [38]:
df=pd.read_csv('/content/100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [39]:
def tockenizer(text):
  text=text.lower()
  text=text.replace(".","")
  text=text.replace("?","")
  text=text.replace(",","")
  text=text.replace("!","")
  return text.split()

In [40]:
vocab={'UNK':0}
def build_vocab(row):
  tockenize_question=tockenizer(row['question'])
  tockenize_answer=tockenizer(row['answer'])
  merge=tockenize_question+tockenize_answer
  for word in merge:
    if word not in vocab:
      vocab[word]=len(vocab)


In [41]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [42]:

def word_to_numeric(text,vocab):
  text_indexing=[]
  for token in tockenizer(text):
    if token not in vocab:
      text_indexing.append(vocab['UNK'])
    else:
      text_indexing.append(vocab[token])
  return text_indexing


In [43]:
class QADataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab

  def __len__(self):
    return len(self.df)

  def __getitem__(self, index):
    row=self.df.iloc[index]
    question=word_to_numeric(row['question'],self.vocab)
    answer=word_to_numeric(row['answer'],self.vocab)
    return torch.tensor(question),torch.tensor(answer)

In [44]:
dataset=QADataset(df,vocab)

In [45]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [46]:
class SimpleRNN(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embeding=nn.Embedding(vocab_size,embedding_dim=50)  ##convert vectorise
    self.rnn=nn.RNN(50,64,batch_first=True) ## connect to rnn
    self.fc=nn.Linear(64,vocab_size) ## connect to fully connected layer
  def forward(self,question):
    embeded_question=self.embeding(question)
    hiden,final=self.rnn(embeded_question)
    output=self.fc(final.squeeze(0))
    return output

In [47]:
learning_rate=0.001
epochs=20
model=SimpleRNN(len(vocab))
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [48]:
for epoch in range(epochs):
  total_loss=0.0
  for q,a in dataloader:
    optimizer.zero_grad()
    output=model(q)
    loss=criterion(output,a[0])
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  print(f"epoch:{epoch+1}|loss:{total_loss:4f}")

epoch:1|loss:522.337133
epoch:2|loss:457.054327
epoch:3|loss:381.818763
epoch:4|loss:315.690936
epoch:5|loss:262.337934
epoch:6|loss:213.035933
epoch:7|loss:170.015219
epoch:8|loss:132.380450
epoch:9|loss:101.449546
epoch:10|loss:77.720920
epoch:11|loss:60.110361
epoch:12|loss:47.017275
epoch:13|loss:37.388946
epoch:14|loss:30.048291
epoch:15|loss:24.534837
epoch:16|loss:20.354161
epoch:17|loss:17.149160
epoch:18|loss:14.536945
epoch:19|loss:12.300500
epoch:20|loss:10.597899


In [58]:
def predict(model,question,threshold=0.5):
  convert_numeric=word_to_numeric(question,vocab)
  convert_tensor=torch.tensor(convert_numeric).unsqueeze(0)
  output=model(convert_tensor)
  probs=torch.nn.functional.softmax(output,dim=1)
  value,index=torch.max(probs,dim=1)
  if value<threshold:
    print("I don't know")
  else:
    print(list(vocab.keys())[index])

In [62]:
predict(model,"What is the capital of Germany?")

berlin
